# GraphRAG: Knowledge Graph-Enhanced RAG with Mistral AI

<a href="https://colab.research.google.com/github/mistralai/cookbook/blob/main/mistral/rag/graphrag_knowledge_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook demonstrates how to build a **Graph RAG** pipeline from scratch using only the Mistral AI SDK. Unlike traditional vector-based RAG, GraphRAG extracts entities and relationships from documents to build a knowledge graph, enabling multi-hop reasoning and more precise retrieval.

**What you'll learn:**
- Extract entities and relationships using Mistral Large's structured outputs
- Build a knowledge graph with NetworkX (no external DB required)
- Combine vector similarity search (FAISS) with graph traversal for hybrid retrieval
- Compare standard RAG vs GraphRAG on multi-hop questions

## Installation

In [ ]:
!pip install mistralai==1.5.0 networkx==3.4.2 faiss-cpu==1.9.0.post1 matplotlib==3.9.4 numpy==1.26.4

## Setup & API Key

In [ ]:
import json
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from typing import List, Optional
from pydantic import BaseModel, Field
from getpass import getpass
from mistralai import Mistral
import faiss

api_key = getpass("Type your API Key")
client = Mistral(api_key=api_key)

## Step 1: Sample Knowledge Base

We use a set of interconnected paragraphs about a fictional technology company to demonstrate multi-hop reasoning.

In [ ]:
documents = [
    "QuantumCore Technologies was founded in 2019 by Dr. Elena Martinez, a former researcher at MIT's CSAIL lab. The company is headquartered in San Francisco and specializes in quantum-enhanced machine learning algorithms.",
    "Dr. Elena Martinez previously collaborated with Prof. James Liu at Stanford on quantum error correction protocols. Prof. Liu later joined QuantumCore's advisory board in 2021.",
    "QuantumCore's flagship product, QML-Engine v3.0, was launched in March 2024. It integrates quantum computing primitives with classical deep learning frameworks like PyTorch and JAX.",
    "In September 2024, QuantumCore announced a partnership with NovaStar Pharma to apply quantum ML to drug discovery. NovaStar's CEO, Dr. Raj Patel, called it 'a paradigm shift in computational biology.'",
    "NovaStar Pharma, based in Boston, was founded in 2015 by Dr. Raj Patel and Dr. Sarah Kim. The company focuses on AI-driven drug discovery and has 3 drugs in Phase II clinical trials.",
    "QuantumCore raised $150M in Series C funding led by Horizon Ventures in January 2025. The valuation reached $2.1B, making it one of the most valuable quantum computing startups.",
    "Horizon Ventures, based in Hong Kong, is known for backing deep tech companies. Their portfolio includes 12 quantum computing and 8 AI companies across Asia and North America.",
    "Dr. Sarah Kim left NovaStar in 2023 to join QuantumCore as VP of Life Sciences. She now leads the drug discovery division that partners with her former company."
]

print(f"Knowledge base: {len(documents)} documents")
for i, doc in enumerate(documents):
    print(f"\n[Doc {i}]: {doc[:80]}...")

## Step 2: Entity & Relationship Extraction with Structured Outputs

We define Pydantic models for entities and relationships, then use Mistral Large to extract them from each document.

In [ ]:
class Entity(BaseModel):
    """An entity extracted from text."""
    name: str = Field(description="The canonical name of the entity")
    type: str = Field(description="Entity type: PERSON, ORGANIZATION, PRODUCT, LOCATION, EVENT, or DATE")
    description: str = Field(description="Brief description of the entity based on context")

class Relationship(BaseModel):
    """A relationship between two entities."""
    source: str = Field(description="Name of the source entity")
    target: str = Field(description="Name of the target entity")
    type: str = Field(description="Relationship type: FOUNDED_BY, WORKS_AT, PARTNER_OF, INVESTED_IN, LOCATED_IN, CREATED, or RELATED_TO")
    description: str = Field(description="Brief description of the relationship")

class ExtractionResult(BaseModel):
    """Complete extraction result from a document."""
    entities: List[Entity] = Field(description="List of extracted entities")
    relationships: List[Relationship] = Field(description="List of extracted relationships")

In [ ]:
def extract_from_document(doc: str, doc_id: int) -> ExtractionResult:
    """Extract entities and relationships from a single document using Mistral Large."""
    
    prompt = f"""Analyze the following text and extract all entities and their relationships.

Text: {doc}

Extract:
1. All named entities (people, organizations, products, locations, dates)
2. All relationships between entities

Be thorough but precise. Only extract what is explicitly stated or directly implied."""

    response = client.chat.complete(
        model="mistral-large-latest",
        messages=[{"role": "user", "content": prompt}],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "extraction",
                "schema": ExtractionResult.model_json_schema()
            }
        }
    )
    
    result = ExtractionResult.model_validate_json(response.choices[0].message.content)
    print(f"[Doc {doc_id}] Extracted {len(result.entities)} entities, {len(result.relationships)} relationships")
    return result

# Extract from all documents
all_extractions = []
for i, doc in enumerate(documents):
    extraction = extract_from_document(doc, i)
    all_extractions.append(extraction)

## Step 3: Build the Knowledge Graph

We merge all extracted entities and relationships into a single NetworkX directed graph.

In [ ]:
G = nx.DiGraph()

entity_registry = {}
for extraction in all_extractions:
    for entity in extraction.entities:
        name = entity.name.strip()
        if name not in entity_registry:
            entity_registry[name] = entity
            G.add_node(name, type=entity.type, description=entity.description)

for extraction in all_extractions:
    for rel in extraction.relationships:
        source = rel.source.strip()
        target = rel.target.strip()
        if source in entity_registry and target in entity_registry:
            G.add_edge(source, target, type=rel.type, description=rel.description)

print(f"Knowledge Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# Count entity types
from collections import Counter
type_counts = Counter(d['type'] for _, d in G.nodes(data=True))
print(f"Entity types: {dict(type_counts)}")

## Step 4: Visualize the Knowledge Graph

In [ ]:
plt.figure(figsize=(14, 10))

# Color nodes by type
color_map = {
    "PERSON": "#FF6B6B",
    "ORGANIZATION": "#4ECDC4",
    "PRODUCT": "#45B7D1",
    "LOCATION": "#96CEB4",
    "EVENT": "#FFEAA7",
    "DATE": "#DDA0DD"
}
node_colors = [color_map.get(G.nodes[n].get("type", ""), "#CCCCCC") for n in G.nodes()]

pos = nx.spring_layout(G, k=2, iterations=50, seed=42)
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=800, alpha=0.9)
nx.draw_networkx_labels(G, pos, font_size=7, font_weight="bold")
nx.draw_networkx_edges(G, pos, edge_color="#888888", arrows=True, arrowsize=15, alpha=0.6)

# Edge labels
edge_labels = {(u, v): d["type"] for u, v, d in G.edges(data=True)}
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=6, alpha=0.7)

# Legend
legend_elements = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=c, markersize=10, label=t) 
                   for t, c in color_map.items() if any(G.nodes[n].get("type") == t for n in G.nodes())]
plt.legend(handles=legend_elements, loc="upper left", fontsize=9)

plt.title("Knowledge Graph \u2014 Extracted with Mistral Large", fontsize=14)
plt.axis("off")
plt.tight_layout()
plt.show()

## Step 5: Document Embeddings with Mistral Embed

We create vector embeddings for each document using Mistral's embedding model, stored in a FAISS index for similarity search.

In [ ]:
def get_embeddings(texts: List[str]) -> np.ndarray:
    """Get embeddings for a list of texts using Mistral Embed."""
    response = client.embeddings.create(
        model="mistral-embed",
        inputs=texts
    )
    return np.array([item.embedding for item in response.data], dtype="float32")

# Embed all documents
doc_embeddings = get_embeddings(documents)
print(f"Embeddings shape: {doc_embeddings.shape}")  # (8, 1024)

# Build FAISS index
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(doc_embeddings)
print(f"FAISS index: {index.ntotal} vectors, dimension {dimension}")

## Step 6: Hybrid Retrieval \u2014 Vector Search + Graph Traversal

This is the core of GraphRAG: we combine FAISS vector similarity with knowledge graph traversal.

1. **Vector retrieval**: Find top-k similar documents via FAISS
2. **Entity extraction**: Extract entities from the query
3. **Graph traversal**: Follow relationships from query entities (BFS, 2 hops)
4. **Merge & deduplicate**: Combine both document sets

In [ ]:
def vector_retrieval(query: str, k: int = 3) -> List[int]:
    """Retrieve top-k documents by vector similarity."""
    query_embedding = get_embeddings([query])
    distances, indices = index.search(query_embedding, k)
    return indices[0].tolist()

def extract_query_entities(query: str) -> List[str]:
    """Extract entity names from the query using Mistral."""
    response = client.chat.complete(
        model="mistral-large-latest",
        messages=[{"role": "user", "content": f"Extract all entity names from this query. Return a JSON list of strings.\n\nQuery: {query}"}],
        response_format={"type": "json_object"}
    )
    data = json.loads(response.choices[0].message.content)
    # Handle various response formats
    if isinstance(data, list):
        return data
    if isinstance(data, dict):
        for key in ["entities", "names", "entity_names"]:
            if key in data and isinstance(data[key], list):
                return data[key]
    return []

def graph_retrieval(query_entities: List[str], max_hops: int = 2) -> set:
    """Retrieve document indices by traversing the knowledge graph from query entities."""
    relevant_nodes = set()
    
    # Find matching nodes in the graph
    for entity in query_entities:
        for node in G.nodes():
            if entity.lower() in node.lower() or node.lower() in entity.lower():
                relevant_nodes.add(node)
                # BFS traversal up to max_hops
                for hop in range(1, max_hops + 1):
                    neighbors = set()
                    for n in list(relevant_nodes):
                        neighbors.update(G.successors(n))
                        neighbors.update(G.predecessors(n))
                    relevant_nodes.update(neighbors)
    
    # Map nodes back to documents that mention them
    relevant_doc_indices = set()
    for idx, doc in enumerate(documents):
        for node in relevant_nodes:
            if node.lower() in doc.lower():
                relevant_doc_indices.add(idx)
    
    return relevant_doc_indices

def hybrid_retrieval(query: str, k_vector: int = 3, max_hops: int = 2) -> List[int]:
    """Combine vector and graph retrieval."""
    # Vector retrieval
    vector_docs = set(vector_retrieval(query, k=k_vector))
    
    # Graph retrieval
    query_entities = extract_query_entities(query)
    print(f"  Query entities: {query_entities}")
    graph_docs = graph_retrieval(query_entities, max_hops=max_hops)
    
    # Merge
    all_docs = vector_docs | graph_docs
    print(f"  Vector: {sorted(vector_docs)} | Graph: {sorted(graph_docs)} | Merged: {sorted(all_docs)}")
    
    return sorted(all_docs)

## Step 7: RAG vs GraphRAG \u2014 Comparison

We compare standard vector-only RAG with our hybrid GraphRAG on multi-hop questions.

In [ ]:
def answer_with_context(query: str, doc_indices: List[int], method_name: str) -> str:
    """Generate an answer using retrieved documents as context."""
    context = "\n\n".join([f"[Doc {i}]: {documents[i]}" for i in doc_indices])
    
    response = client.chat.complete(
        model="mistral-large-latest",
        messages=[
            {"role": "system", "content": "Answer the question based only on the provided context. If the context doesn't contain enough information, say so."},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"}
        ]
    )
    return response.choices[0].message.content

# Multi-hop questions that require connecting information across documents
test_questions = [
    "What is the connection between Dr. Sarah Kim and QuantumCore's drug discovery partnership?",
    "Who are the academic collaborators behind QuantumCore, and what institutions are they from?",
    "Trace the funding and valuation history of QuantumCore, including who led the investment.",
    "How are NovaStar Pharma and Horizon Ventures connected through QuantumCore?"
]

print("=" * 80)
print("RAG vs GraphRAG COMPARISON")
print("=" * 80)

results = []
for q in test_questions:
    print(f"\n{'='*80}")
    print(f"QUESTION: {q}")
    
    # Standard RAG (vector only)
    print(f"\n--- Standard RAG (top-3 vector) ---")
    vector_docs = vector_retrieval(q, k=3)
    print(f"  Retrieved docs: {vector_docs}")
    rag_answer = answer_with_context(q, vector_docs, "RAG")
    print(f"  Answer: {rag_answer[:200]}...")
    
    # GraphRAG (hybrid)
    print(f"\n--- GraphRAG (hybrid) ---")
    hybrid_docs = hybrid_retrieval(q, k_vector=3, max_hops=2)
    graphrag_answer = answer_with_context(q, hybrid_docs, "GraphRAG")
    print(f"  Answer: {graphrag_answer[:200]}...")
    
    results.append({
        "question": q,
        "rag_docs": len(vector_docs),
        "graphrag_docs": len(hybrid_docs),
        "extra_docs": len(set(hybrid_docs) - set(vector_docs))
    })

## Results Summary

In [ ]:
print("\n=== RETRIEVAL COMPARISON ===\n")
print(f"{'Question':<70} {'RAG docs':>10} {'GraphRAG docs':>14} {'Extra from graph':>17}")
print("-" * 115)
for r in results:
    print(f"{r['question'][:68]:<70} {r['rag_docs']:>10} {r['graphrag_docs']:>14} {r['extra_docs']:>17}")

avg_extra = sum(r['extra_docs'] for r in results) / len(results)
print(f"\nAverage extra documents from graph traversal: {avg_extra:.1f}")
print(f"Graph nodes: {G.number_of_nodes()} | Graph edges: {G.number_of_edges()}")

## Summary

This notebook demonstrated a complete GraphRAG pipeline built from scratch with Mistral AI:

| Component | Implementation |
|-----------|---------------|
| Entity extraction | Mistral Large + Pydantic structured outputs |
| Knowledge graph | NetworkX DiGraph with typed nodes and edges |
| Vector index | FAISS IndexFlatL2 with Mistral Embed (1024d) |
| Hybrid retrieval | Vector similarity + BFS graph traversal (2 hops) |
| Generation | Mistral Large with retrieved context |

### Considerations:
- **Scaling**: For larger knowledge bases, consider Neo4j or ArangoDB instead of NetworkX
- **Entity resolution**: Add fuzzy matching or embedding-based deduplication for entity names
- **Graph updates**: Implement incremental extraction for new documents without rebuilding the full graph
- **Evaluation**: Compare retrieval metrics (precision@k, recall@k) between RAG and GraphRAG systematically
- **Production**: Add caching for embeddings and extraction results to reduce API costs